In [1]:
# Kontrak Notebook 00 — dicetak verbatim sebagai output pertama (wajib, dari Rancangan EDA v3 Final).
KONTRAK_00 = """- Sydney: 08:00–16:00 waktu lokal `Australia/Sydney` secara DST-aware, konsisten dengan seluruh sesi (keputusan MEDIUM-14 opsi A; kontradiksi v2 antara angka 21–06 UTC dan "DST diabaikan" dicabut).
- Prior-day H/L: primer = bucket D1 kalender-broker (batas tengah malam broker: 21:00 UTC summer, 22:00 UTC winter), mencakup ekor 00:00–03:00 wall sesi sebelumnya; bucket Sabtu/Minggu-wall tidak ada; sekunder = hari-UTC yang dihitung ulang; kedua definisi wajib sepakat, perbedaan dilaporkan sebagai sensitivitas (keputusan MEDIUM-15 + errata E1/E2).
- Statistik regime: granularitas harian; effect-size wajib (beda median dengan CI, Cliff's delta); p-value tanpa effect-size dilarang disimpulkan; catatan keterbatasan clustering wajib; block bootstrap sirkular (block 20 hari dagang, 10.000 resample) hanya bila hasil marginal; setiap tabel uji mencantumkan n-A/n-B/n-C; untuk strata B (n≈130 hari dagang) hasil negatif dibaca "tidak terdeteksi pada n ini" (keputusan MEDIUM-16 + errata E3).
- Fragmen spread: langkah 4b terpisah dengan deliverable fragmen + diff-vs-config + catatan dampak; tanpa putusan user tidak diterapkan (keputusan MEDIUM-17).
- tzdata: `tzdata==2026.5` terverifikasi di venv proyek, ditambahkan ke requirements + CHANGELOG atas persetujuan user (LOW-19/25).
- Missing-data: drop dilaporkan jumlahnya; NaN+flag untuk warmup; window-kosong diinvestigasi; nol imputasi (LOW-20).
- Kalender libur: di luar scope; missing-event tercatat di laporan (LOW-21).
- Prior sesi ganda untuk dikalibrasi M1–M4: pangsa volume global dan prior khusus-USDJPY London 66 > NewYork 59 > Tokyo 51 pip; angka S5/S6 berlabel sumber sekunder (LOW-17/18).
- Strata dan hitungan kunci: A + B + C = 1.347.847 bar; B = 185.248 bar = 13,74%; C = 26.198 bar = 1,94% (5 + 14 hari); 2025 tahun referensi bersih."""
print(KONTRAK_00)

- Sydney: 08:00–16:00 waktu lokal `Australia/Sydney` secara DST-aware, konsisten dengan seluruh sesi (keputusan MEDIUM-14 opsi A; kontradiksi v2 antara angka 21–06 UTC dan "DST diabaikan" dicabut).
- Prior-day H/L: primer = bucket D1 kalender-broker (batas tengah malam broker: 21:00 UTC summer, 22:00 UTC winter), mencakup ekor 00:00–03:00 wall sesi sebelumnya; bucket Sabtu/Minggu-wall tidak ada; sekunder = hari-UTC yang dihitung ulang; kedua definisi wajib sepakat, perbedaan dilaporkan sebagai sensitivitas (keputusan MEDIUM-15 + errata E1/E2).
- Statistik regime: granularitas harian; effect-size wajib (beda median dengan CI, Cliff's delta); p-value tanpa effect-size dilarang disimpulkan; catatan keterbatasan clustering wajib; block bootstrap sirkular (block 20 hari dagang, 10.000 resample) hanya bila hasil marginal; setiap tabel uji mencantumkan n-A/n-B/n-C; untuk strata B (n≈130 hari dagang) hasil negatif dibaca "tidak terdeteksi pada n ini" (keputusan MEDIUM-16 + errata E3).
- Fragme

# EDA USDJPY — Notebook 00: Setup & Kontrak

Fondasi seluruh rantai EDA v3. Semua yang dikunci di sini dipakai notebook 01–05 tanpa perubahan.

**Isi**
1. Cetak Kontrak 00 verbatim (output pertama).
2. Audit parquet + konfigurasi (fase setup).
3. Audit hitungan strata A/B/C terhadap 1.347.847 bar.
4. Tabel pemetaan sesi primer (DST-aware) vs cross-tab (UTC-tetap) untuk contoh winter & summer.
5. Matriks ketersediaan kolom per timeframe + audit missing-data (nol imputasi).
6. Penguncian definisi return/volatilitas/swing/level + aturan anti look-ahead.
7. Preregistrasi protokol placebo Q6 (level) — ditulis sebelum hasil apa pun dilihat.
8. Ringkasan audit.

**Aturan eksekusi**: notebook dieksekusi ulang top-to-bottom dengan venv kanonik `venv/`; output disimpan; tidak ada dump data mentah; notebook TIDAK menulis config apa pun.

In [2]:
import importlib.metadata
import sys
from datetime import UTC, date, datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    """Cari root proyek (pyproject.toml + src/) dari cwd ke atas."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("PROJECT_ROOT tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.broker_time import BrokerSchedule, utc_series, wall_clock_series  # noqa: E402
from src.utils.config import load_config  # noqa: E402

print("PROJECT_ROOT :", PROJECT_ROOT)
print("python       :", sys.version.split()[0])
for package in ("pandas", "numpy", "pyarrow", "tzdata"):
    print(f"  {package:9s}: {importlib.metadata.version(package)}")
assert importlib.metadata.version("tzdata") == "2026.5", "tzdata harus 2026.5 (Kontrak 00)"

PROJECT_ROOT : D:\ANN
python       : 3.12.8
  pandas   : 2.2.3
  numpy    : 1.26.4
  pyarrow  : 17.0.0
  tzdata   : 2026.5


In [3]:
config = load_config("data")

SYMBOL = config["mt5"]["symbol"]
SCHEDULE = BrokerSchedule.from_config(config["mt5"])
ANOMALY_WINDOWS = tuple((w["start"], w["end"]) for w in config["validation"]["anomaly_windows"])
SPREAD_ASSUMPTIONS = {
    int(year): float(value) for year, value in config["spread"]["by_year"].items()
}

print("Simbol          :", SYMBOL)
print(
    "Offset broker   : standard",
    SCHEDULE.standard_offset,
    "s | daylight",
    SCHEDULE.daylight_offset,
    "s",
)
print("Jendela DST     :", len(SCHEDULE.daylight_windows), "jendela (2023-2026)")
print("Anomali windows :", ANOMALY_WINDOWS)
print("Spread (asumsi) :", SPREAD_ASSUMPTIONS)
print()
print("CATATAN: spread di config = ASUMSI sampai fragmen langkah 4b disetujui user.")
assert len(ANOMALY_WINDOWS) == 2, "harus tepat 2 window anomali (Klenz-verified)"

Simbol          : USDJPY
Offset broker   : standard 7200 s | daylight 10800 s
Jendela DST     : 4 jendela (2023-2026)
Anomali windows : (('2024-10-28', '2024-11-01'), ('2026-03-10', '2026-03-27'))
Spread (asumsi) : {2023: 3.0, 2024: 1.5, 2025: 1.5, 2026: 1.5}

CATATAN: spread di config = ASUMSI sampai fragmen langkah 4b disetujui user.


## Stratifikasi (dikunci di sini)

Tiga strata dipisahkan sejak awal dan tidak boleh bocor satu sama lain:

- **A = utama** — sisa bar setelah B dan C dieksklusi (target 1.136.401 bar).
- **B = H1-2023** — 2023-01-01 s.d. 2023-06-30 (regime data berbeda, temuan SA-03; target 185.248 bar = 13,74%).
- **C = window anomali** — 2024-10-28..11-01 dan 2026-03-10..03-27 (target 26.198 bar = 1,94%).

Definisi operasional: pemetaan bar → strata memakai **tanggal dinding broker** (`broker_time_raw`),
bukan tanggal UTC. Untuk C ini krusial — gejalanya artefak jam dinding broker dan offset jendela
masih BELUM TERVERIFIKASI; untuk B kedua kalender memberi angka sama (diverifikasi di bawah).

In [4]:
M1_COLUMNS = ["time", "broker_time_raw"]
m1 = pd.concat(
    [
        pd.read_parquet(
            PROJECT_ROOT / "data/processed/M1" / f"usdjpy_m1_{year}.parquet", columns=M1_COLUMNS
        )
        for year in (2023, 2024, 2025, 2026)
    ],
    ignore_index=True,
)
m1["wall"] = wall_clock_series(m1["broker_time_raw"])
m1["utc"] = utc_series(m1["broker_time_raw"], SCHEDULE)

wall_date = m1["wall"].dt.normalize()

b_mask = (wall_date >= pd.Timestamp("2023-01-01")) & (wall_date <= pd.Timestamp("2023-06-30"))
c_mask = pd.Series(False, index=m1.index)
c_per_window = []
for start, end in ANOMALY_WINDOWS:
    in_window = (wall_date >= pd.Timestamp(start)) & (wall_date <= pd.Timestamp(end))
    c_mask |= in_window
    c_per_window.append((start, end, int(in_window.sum()), int(wall_date[in_window].nunique())))
a_mask = ~(b_mask | c_mask)

TOTAL = len(m1)
n_a, n_b, n_c = int(a_mask.sum()), int(b_mask.sum()), int(c_mask.sum())
days_a = int(wall_date[a_mask].nunique())
days_b = int(wall_date[b_mask].nunique())
days_c = int(wall_date[c_mask].nunique())
bars_2025_anomaly = int((c_mask & (wall_date.dt.year == 2025)).sum())

print(f"TOTAL: {TOTAL:,} bar | {wall_date.nunique()} hari broker")
print()
print(f"{'Strata':6} {'Bar':>10} {'Pangsa':>8} {'Hari':>6}")
for name, bars, days in [("A", n_a, days_a), ("B", n_b, days_b), ("C", n_c, days_c)]:
    print(f"{name:6} {bars:>10,} {bars / TOTAL:>7.2%} {days:>6}")
total_label = "A+B+C"
print(f"{total_label:6} {n_a + n_b + n_c:>10,}")

print()
for start, end, bars, days in c_per_window:
    print(f"C window {start}..{end}: {bars:,} bar, {days} hari broker")

assert n_b == 185_248, f"B harus 185.248, dapat {n_b}"
assert n_c == 26_198, f"C harus 26.198, dapat {n_c}"
assert n_a == 1_136_401, f"A harus 1.136.401, dapat {n_a}"
assert n_a + n_b + n_c == TOTAL == 1_347_847
assert not (b_mask & c_mask).any(), "B dan C tidak boleh tumpang tindih"
assert days_b == 130, f"H1-2023 harus 130 hari berbar, dapat {days_b}"
assert [(bars, days) for _, _, bars, days in c_per_window] == [(6_899, 5), (19_299, 14)]
assert bars_2025_anomaly == 0, "2025 harus bersih dari anomali"
assert wall_date.nunique() == days_a + days_b + days_c, "hari broker harus terpartisi"
print()
print("PASS: strata mereproduksi angka Lapisan 1; 2025 bersih; partisi hari utuh.")

TOTAL: 1,347,847 bar | 974 hari broker

Strata        Bar   Pangsa   Hari
A       1,136,401  84.31%    825
B         185,248  13.74%    130
C          26,198   1.94%     19
A+B+C   1,347,847

C window 2024-10-28..2024-11-01: 6,899 bar, 5 hari broker
C window 2026-03-10..2026-03-27: 19,299 bar, 14 hari broker

PASS: strata mereproduksi angka Lapisan 1; 2025 bersih; partisi hari utuh.


## Definisi Sesi (dikunci di sini)

Dua pelabelan dihitung; **DST-aware adalah primer**, UTC-tetap hanya cross-tab pembanding:

| Sesi | Zona IANA | Window lokal (primer) | Cross-tab UTC-tetap |
|---|---|---|---|
| Tokyo | Asia/Tokyo | 09:00–18:00 | 00:00–09:00 |
| London | Europe/London | 08:00–16:00 | 08:00–16:00 |
| New York | America/New_York | 08:00–17:00 | 13:00–22:00 |
| Sydney | Australia/Sydney | 08:00–16:00 | 21:00–06:00 |

- Flag per bar non-eksklusif (`is_tokyo`, `is_london`, `is_ny`, `is_sydney`) karena sesi tumpang tindih.
- Label eksklusif untuk tabulasi: prioritas **London > New York > Tokyo > Sydney > Off**
  (dari prior volume global + prior USDJPY S5/S6 — sumber sekunder, dikalibrasi M1–M4 di EDA-02).
- Temuan sesi yang tidak bertahan di kedua pelabelan diklasifikasikan artefak batas, bukan temuan.

In [5]:
SESSION_SPECS = {
    "is_tokyo": {"name": "Tokyo", "zone": "Asia/Tokyo", "start": 9, "end": 18},
    "is_london": {"name": "London", "zone": "Europe/London", "start": 8, "end": 16},
    "is_ny": {"name": "New York", "zone": "America/New_York", "start": 8, "end": 17},
    "is_sydney": {"name": "Sydney", "zone": "Australia/Sydney", "start": 8, "end": 16},
}
SESSION_PRIORITY = [
    ("is_london", "London"),
    ("is_ny", "NewYork"),
    ("is_tokyo", "Tokyo"),
    ("is_sydney", "Sydney"),
]
CROSS_TAB_HOURS = {
    "is_tokyo": (0, 9),
    "is_london": (8, 16),
    "is_ny": (13, 22),
    "is_sydney": (21, 6),
}


def session_flags(utc: pd.Series) -> pd.DataFrame:
    """Flag sesi DST-aware (primer) dari series UTC tz-aware.

    Args:
        utc: Series bertipe datetime64[ns, UTC].

    Returns:
        DataFrame boolean berkolom is_tokyo, is_london, is_ny, is_sydney.
    """
    flags = {}
    for column, spec in SESSION_SPECS.items():
        local_hour = utc.dt.tz_convert(spec["zone"]).dt.hour
        flags[column] = (local_hour >= spec["start"]) & (local_hour < spec["end"])
    return pd.DataFrame(flags, index=utc.index)


def cross_tab_flags(utc: pd.Series) -> pd.DataFrame:
    """Flag sesi UTC-tetap (cross-tab pembanding campuran-musim) dari series UTC."""
    hour = utc.dt.hour
    flags = {}
    for column, (start, end) in CROSS_TAB_HOURS.items():
        if start < end:
            flags[column] = (hour >= start) & (hour < end)
        else:
            flags[column] = (hour >= start) | (hour < end)
    return pd.DataFrame(flags, index=utc.index)


def exclusive_label(flags: pd.DataFrame) -> pd.Series:
    """Label eksklusif: prioritas London > NewYork > Tokyo > Sydney > Off."""
    conditions = [flags[column] for column, _ in SESSION_PRIORITY]
    choices = [label for _, label in SESSION_PRIORITY]
    return pd.Series(
        np.select(conditions, choices, default="Off"), index=flags.index, name="session_label"
    )


print("Sesi primer (DST-aware):")
for spec in SESSION_SPECS.values():
    print(f"  {spec['name']:9s} {spec['start']:02d}:00-{spec['end']:02d}:00 {spec['zone']}")
priority_text = " > ".join(label for _, label in SESSION_PRIORITY) + " > Off"
print("Prioritas label eksklusif:", priority_text)
print("Cross-tab UTC-tetap     :", CROSS_TAB_HOURS)

Sesi primer (DST-aware):
  Tokyo     09:00-18:00 Asia/Tokyo
  London    08:00-16:00 Europe/London
  New York  08:00-17:00 America/New_York
  Sydney    08:00-16:00 Australia/Sydney
Prioritas label eksklusif: London > NewYork > Tokyo > Sydney > Off
Cross-tab UTC-tetap     : {'is_tokyo': (0, 9), 'is_london': (8, 16), 'is_ny': (13, 22), 'is_sydney': (21, 6)}


In [6]:
WINTER_SAMPLE = date(2025, 1, 15)
SUMMER_SAMPLE = date(2025, 7, 15)


def utc_window(
    day: date, zone_name: str, start_hour: int, end_hour: int
) -> tuple[datetime, datetime]:
    """Konversi window jam lokal menjadi window UTC untuk tanggal contoh."""
    zone = ZoneInfo(zone_name)
    start = datetime(day.year, day.month, day.day, start_hour, tzinfo=zone).astimezone(UTC)
    end = datetime(day.year, day.month, day.day, end_hour, tzinfo=zone).astimezone(UTC)
    return start, end


rows = []
for column, spec in SESSION_SPECS.items():
    winter_start, winter_end = utc_window(WINTER_SAMPLE, spec["zone"], spec["start"], spec["end"])
    summer_start, summer_end = utc_window(SUMMER_SAMPLE, spec["zone"], spec["start"], spec["end"])
    plus_winter = "+1" if winter_end.date() > winter_start.date() else ""
    plus_summer = "+1" if summer_end.date() > summer_start.date() else ""
    cross_start, cross_end = CROSS_TAB_HOURS[column]
    rows.append(
        {
            "sesi": spec["name"],
            "zona": spec["zone"],
            "lokal": f"{spec['start']:02d}:00-{spec['end']:02d}:00",
            "UTC winter (15 Jan 2025)": f"{winter_start:%H:%M}-{winter_end:%H:%M}{plus_winter}",
            "UTC summer (15 Jul 2025)": f"{summer_start:%H:%M}-{summer_end:%H:%M}{plus_summer}",
            "cross-tab UTC": f"{cross_start:02d}:00-{cross_end:02d}:00",
        }
    )
mapping = pd.DataFrame(rows)
print(mapping.to_string(index=False))
print()
print(
    "Catatan: tanda +1 = batas akhir jatuh di hari UTC berikutnya (window menyeberang tengah malam UTC)."
)

expected = {
    ("London", "winter"): ("08:00", "16:00"),
    ("London", "summer"): ("07:00", "15:00"),
    ("New York", "winter"): ("13:00", "22:00"),
    ("New York", "summer"): ("12:00", "21:00"),
    ("Sydney", "winter"): ("21:00", "05:00"),
    ("Sydney", "summer"): ("22:00", "06:00"),
}
for (name, season), (want_start, want_end) in expected.items():
    spec = next(spec for spec in SESSION_SPECS.values() if spec["name"] == name)
    day = WINTER_SAMPLE if season == "winter" else SUMMER_SAMPLE
    start, end = utc_window(day, spec["zone"], spec["start"], spec["end"])
    assert (f"{start:%H:%M}", f"{end:%H:%M}") == (want_start, want_end), (name, season)
for day in (WINTER_SAMPLE, SUMMER_SAMPLE):
    start, end = utc_window(day, "Asia/Tokyo", 9, 18)
    assert (f"{start:%H:%M}", f"{end:%H:%M}") == ("00:00", "09:00")
print("PASS: konversi winter/summer terverifikasi; Tokyo konstan 00:00-09:00 UTC.")

    sesi             zona       lokal UTC winter (15 Jan 2025) UTC summer (15 Jul 2025) cross-tab UTC
   Tokyo       Asia/Tokyo 09:00-18:00              00:00-09:00              00:00-09:00   00:00-09:00
  London    Europe/London 08:00-16:00              08:00-16:00              07:00-15:00   08:00-16:00
New York America/New_York 08:00-17:00              13:00-22:00              12:00-21:00   13:00-22:00
  Sydney Australia/Sydney 08:00-16:00            21:00-05:00+1            22:00-06:00+1   21:00-06:00

Catatan: tanda +1 = batas akhir jatuh di hari UTC berikutnya (window menyeberang tengah malam UTC).
PASS: konversi winter/summer terverifikasi; Tokyo konstan 00:00-09:00 UTC.


In [7]:
sample = m1[wall_date.isin([pd.Timestamp(WINTER_SAMPLE), pd.Timestamp(SUMMER_SAMPLE)])].copy()
flags = session_flags(sample["utc"])
cross_flags = cross_tab_flags(sample["utc"])
sample = sample.join(flags)
sample["label"] = exclusive_label(flags)
sample["label_cross"] = exclusive_label(cross_flags)

for day in (WINTER_SAMPLE, SUMMER_SAMPLE):
    day_rows = sample[sample["wall"].dt.normalize() == pd.Timestamp(day)]
    print(f"{day}: {len(day_rows)} bar M1")
    print("  label primer:", day_rows["label"].value_counts().to_dict())
    print("  label cross :", day_rows["label_cross"].value_counts().to_dict())

for column, label in SESSION_PRIORITY:
    labeled = sample.loc[sample["label"] == label, column]
    assert bool(labeled.all()), f"bar berlabel {label} wajib punya {column}=True"
assert set(sample["label"].unique()) <= {"London", "NewYork", "Tokyo", "Sydney", "Off"}
overlap_london_ny = int((flags["is_london"] & flags["is_ny"]).sum())
overlap_tokyo_london = int((flags["is_tokyo"] & flags["is_london"]).sum())
print()
print(
    "Overlap 2 hari contoh — London+NY:",
    overlap_london_ny,
    "bar | Tokyo+London:",
    overlap_tokyo_london,
    "bar",
)
assert overlap_london_ny > 0 and overlap_tokyo_london > 0
print("PASS: pelabelan eksklusif konsisten dengan flag non-eksklusif; overlap terdeteksi.")

2025-01-15: 1380 bar M1
  label primer: {'Tokyo': 480, 'London': 480, 'NewYork': 360, 'Sydney': 60}
  label cross : {'Tokyo': 480, 'London': 480, 'NewYork': 360, 'Sydney': 60}
2025-07-15: 1380 bar M1
  label primer: {'London': 480, 'Tokyo': 420, 'NewYork': 360, 'Sydney': 120}
  label cross : {'Tokyo': 480, 'London': 480, 'NewYork': 300, 'Sydney': 120}

Overlap 2 hari contoh — London+NY: 360 bar | Tokyo+London: 180 bar
PASS: pelabelan eksklusif konsisten dengan flag non-eksklusif; overlap terdeteksi.


## Matriks Ketersediaan Kolom & Audit Missing-Data

- Sumber tunggal: parquet `data/processed/<TF>/usdjpy_<tf>_<tahun>.parquet` (2023–2026).
- Dipakai EDA: M1 (primer), M5/M15 (konfirmasi), H1 (konteks), D1 (prior-day H/L). M30/H4 ada tetapi tidak dipakai.
- Catatan schema: kolom `time` tersimpan sebagai **string ISO-8601 UTC** (bukan timestamp parquet).
  Notebook memakai `utc_series(broker_time_raw)` sebagai sumber waktu; kesetaraan dengan parse `time`
  diverifikasi penuh pada M1 2025.
- Missing-data: **nol imputasi** (Kontrak 00). Audit di bawah melaporkan NaN apa adanya; drop apa pun
  di analisis nanti wajib dilaporkan jumlahnya.

In [8]:
EDA_TFS = ["M1", "M5", "M15", "H1", "D1"]
EXTRA_TFS = ["M30", "H4"]
YEARS = (2023, 2024, 2025, 2026)

inventory_rows = []
nan_report = {}
columns_by_tf = {}
m1_full = None
for tf in EDA_TFS + EXTRA_TFS:
    files = sorted((PROJECT_ROOT / "data/processed" / tf).glob(f"usdjpy_{tf.lower()}_*.parquet"))
    assert [path.stem.split("_")[-1] for path in files] == [
        str(year) for year in YEARS
    ], f"{tf}: file tahun tidak lengkap"
    frames = [pd.read_parquet(path) for path in files]
    combined = pd.concat(frames, ignore_index=True)
    columns_by_tf[tf] = list(combined.columns)
    bad = {column: int(count) for column, count in combined.isna().sum().items() if count}
    nan_report[tf] = bad
    per_year = {
        path.stem.split("_")[-1]: len(frame) for path, frame in zip(files, frames, strict=False)
    }
    inventory_rows.append(
        {
            "TF": tf,
            "file": len(files),
            "bar": len(combined),
            **per_year,
            "dipakai EDA": "ya" if tf in EDA_TFS else "tidak",
        }
    )
    if tf == "M1":
        m1_full = combined

inventory = pd.DataFrame(inventory_rows)
print(inventory.to_string(index=False))
print()
print("NaN per TF:", {tf: (bad if bad else "none") for tf, bad in nan_report.items()})
nan_ok = all(not bad for bad in nan_report.values())
assert nan_ok, "tidak boleh ada NaN (nol imputasi)"

ALL_COLUMNS = [
    "time",
    "broker_time_raw",
    "open",
    "high",
    "low",
    "close",
    "tick_volume",
    "real_volume",
    "spread_points",
    "spread_pips",
    "spread_pips_mean",
    "spread_pips_max",
    "m1_bars",
]
matrix = pd.DataFrame(
    {
        tf: {column: ("ya" if column in columns_by_tf[tf] else "-") for column in ALL_COLUMNS}
        for tf in EDA_TFS + EXTRA_TFS
    }
)
print()
print("Matriks kolom (ya = tersedia):")
print(matrix.to_string())
for tf in EDA_TFS + EXTRA_TFS:
    for column in (
        "time",
        "broker_time_raw",
        "open",
        "high",
        "low",
        "close",
        "tick_volume",
        "real_volume",
    ):
        assert column in columns_by_tf[tf], f"{tf} kehilangan kolom {column}"
for column in ("spread_points", "spread_pips"):
    assert column in columns_by_tf["M1"], f"M1 kehilangan kolom {column}"
for tf in ["M5", "M15", "H1", "D1"]:
    for column in ("spread_pips_mean", "spread_pips_max", "m1_bars"):
        assert column in columns_by_tf[tf], f"{tf} kehilangan kolom {column}"

m1_2025 = m1_full[m1_full["time"].str.startswith("2025")]
same = (
    utc_series(m1_2025["broker_time_raw"], SCHEDULE).values
    == pd.to_datetime(m1_2025["time"], utc=True).values
)
time_eq_ok = bool(same.all())
print()
print(f"M1 2025: utc_series == parse(time): {time_eq_ok} ({len(m1_2025):,} bar)")
assert time_eq_ok

for year in YEARS:
    d1 = pd.read_parquet(PROJECT_ROOT / "data/processed/D1" / f"usdjpy_d1_{year}.parquet")
    wall = wall_clock_series(d1["broker_time_raw"])
    assert bool((wall == wall.dt.normalize()).all()), f"D1 {year}: label bukan tengah malam broker"
    assert int((wall.dt.dayofweek >= 5).sum()) == 0, f"D1 {year}: ada bucket Sabtu/Minggu"
d1_2024_buckets = int(
    len(pd.read_parquet(PROJECT_ROOT / "data/processed/D1" / "usdjpy_d1_2024.parquet"))
)
d1_ok = d1_2024_buckets == 260
assert d1_ok
print(f"D1: label = 00:00 dinding broker, 0 bucket Sabtu/Minggu, 2024 = {d1_2024_buckets} bucket")
print("PASS: inventory file/kolom/NaN/D1/time-setara terverifikasi.")

 TF  file     bar   2023   2024   2025   2026 dipakai EDA
 M1     4 1347847 362693 358331 356977 269846          ya
 M5     4  269964  72742  71715  71432  54075          ya
M15     4   90041  24294  23909  23812  18026          ya
 H1     4   22513   6074   5979   5953   4507          ya
 D1     4     974    259    260    259    196          ya
M30     4   45024  12148  11956  11906   9014       tidak
 H4     4    5844   1554   1560   1554   1176       tidak

NaN per TF: {'M1': 'none', 'M5': 'none', 'M15': 'none', 'H1': 'none', 'D1': 'none', 'M30': 'none', 'H4': 'none'}

Matriks kolom (ya = tersedia):
                  M1  M5 M15  H1  D1 M30  H4
time              ya  ya  ya  ya  ya  ya  ya
broker_time_raw   ya  ya  ya  ya  ya  ya  ya
open              ya  ya  ya  ya  ya  ya  ya
high              ya  ya  ya  ya  ya  ya  ya
low               ya  ya  ya  ya  ya  ya  ya
close             ya  ya  ya  ya  ya  ya  ya
tick_volume       ya  ya  ya  ya  ya  ya  ya
real_volume       ya  ya  ya  


M1 2025: utc_series == parse(time): True (356,977 bar)
D1: label = 00:00 dinding broker, 0 bucket Sabtu/Minggu, 2024 = 260 bucket
PASS: inventory file/kolom/NaN/D1/time-setara terverifikasi.


## Penguncian Definisi (berlaku notebook 01–05)

| Objek | Definisi terkunci | Catatan |
|---|---|---|
| Return | `r = ln(close_t / close_{t-1})`, close = BID | tanpa P&L; spread = biaya terpisah |
| Volatilitas | sum_r2 dan mean_abs_r per bucket; `tick_volume` proksi; rasio sesi/harian | per jam broker, per sesi, per hari |
| Swing | fraktal k=3 dan k=5 bar per sisi di M5 — keduanya dilaporkan | ukuran dinormalisasi ATR(14) |
| Level | round xx.00 primer + xx.50 alternatif; prior-day H/L ganda (broker/UTC) | placebo wajib (preregistrasi di bawah) |
| Anti look-ahead | semua rolling `.shift(1)`; event-study hanya jendela pasca-event | tidak ada data t+1 di fitur t |
| Missing-data | drop dilaporkan; NaN+flag warmup; nol imputasi | konsisten Kontrak 00 |

In [9]:
RETURN_DEFINITION = "r_t = ln(close_t / close_{t-1}); close = BID"
VOLATILITY_METRICS = ("sum_r2", "mean_abs_r", "tick_volume_proxy", "session_vs_daily_ratio")
SWING_FRACTAL_KS = (3, 5)
SWING_SIZE_NORMALIZER = "ATR(14)"
ROUND_LEVEL_GRID = 0.50
ROUND_LEVELS = "xx.00 (primer) + xx.50 (granularitas alternatif)"
PRIOR_DAY_HL_PRIMARY = "bucket D1 kalender-broker (label 00:00 dinding broker)"
PRIOR_DAY_HL_SECONDARY = "hari-UTC dihitung ulang; wajib sepakat, beda = sensitivitas"
LOOKAHEAD_RULE = "rolling .shift(1); event-study hanya pasca-event"
MISSING_DATA_RULE = "drop dilaporkan; NaN+flag warmup; nol imputasi"

print("Definisi terkunci:")
for name, value in [
    ("Return", RETURN_DEFINITION),
    ("Volatilitas", VOLATILITY_METRICS),
    ("Swing k", SWING_FRACTAL_KS),
    ("Level grid", ROUND_LEVEL_GRID),
    ("Level", ROUND_LEVELS),
    ("Prior-day H/L primer", PRIOR_DAY_HL_PRIMARY),
    ("Prior-day H/L sekunder", PRIOR_DAY_HL_SECONDARY),
    ("Anti look-ahead", LOOKAHEAD_RULE),
    ("Missing-data", MISSING_DATA_RULE),
]:
    print(f"  {name}: {value}")

Definisi terkunci:
  Return: r_t = ln(close_t / close_{t-1}); close = BID
  Volatilitas: ('sum_r2', 'mean_abs_r', 'tick_volume_proxy', 'session_vs_daily_ratio')
  Swing k: (3, 5)
  Level grid: 0.5
  Level: xx.00 (primer) + xx.50 (granularitas alternatif)
  Prior-day H/L primer: bucket D1 kalender-broker (label 00:00 dinding broker)
  Prior-day H/L sekunder: hari-UTC dihitung ulang; wajib sepakat, beda = sensitivitas
  Anti look-ahead: rolling .shift(1); event-study hanya pasca-event
  Missing-data: drop dilaporkan; NaN+flag warmup; nol imputasi


## Preregistrasi Protokol Placebo — Q6 (Level)

Ditulis dan dikunci **sebelum** hasil apa pun dilihat; dijalankan di notebook 04 tanpa tuning.
Parameter di bawah adalah keputusan desain EDA (bukan klaim); user/Klenz dapat meninjau naskah ini
sebelum notebook 04 dieksekusi.

**Level yang diuji**
- Round-number: grid kelipatan 0.50 harga (xx.00 primer; xx.50 granularitas alternatif — diuji terpisah).
- Prior-day high/low: primer = bucket D1 kalender-broker; sekunder = hari-UTC; keduanya dilaporkan dan wajib sepakat.
- Fibonacci: hanya boleh diuji dengan protokol placebo identik ini; tanpa bukti = tanpa klaim.

**Touch & reaksi (M5, BID)**
- Touch: bar M5 dengan `low <= L <= high`.
- theta = 0.5 × ATR(14) M5 dihitung sampai bar t−1 (shift 1; anti look-ahead).
- Sisi datang: `sign(close_{t-1} - L)`; bar dengan `close_{t-1} == L` dilewati.
- `through`: harga menembus L sejauh theta ke arah sisi datang (first passage).
- `away`: harga bergerak menjauh dari L sejauh theta (first passage).
- Hit reversal: `away` terjadi dan (`through` tidak terjadi atau `away` lebih dulu).
- Window k ∈ {6, 12, 24} bar M5; primer k = 12 (1 jam); k lain adalah robustness yang sudah dipreregistrasi.

**Kontrol placebo (dua-duanya protokol identik)**
1. Level acak ber-offset: L' = L + U(5, 25) pip dengan tanda acak per event; jumlah event sama.
2. Prior-day H/L digeser: tiap H/L digeser U(0.25, 1.00) × ATR(14) harian dengan tanda acak independen per level per hari.

**Inferensi**
- Hit-rate level nyata vs distribusi placebo; CI persentil 95% via bootstrap 10.000 resample,
  unit resample = **hari perdagangan** (day-cluster, menghormati dependensi intraday).
- Seed tetap `PLACEBO_SEED = 20261006`; tidak ada penggantian seed.
- Tidak ada p-value tanpa effect-size (aturan AGENTS §20/§21).

In [10]:
PLACEBO_SEED = 20261006
PLACEBO_TIMEFRAME = "M5"
TOUCH_ATR_WINDOW = 14
REACTION_THETA_ATR = 0.5
REACTION_WINDOWS_BARS = (6, 12, 24)
REACTION_WINDOW_PRIMARY = 12
PLACEBO_RANDOM_OFFSET_PIPS = (5.0, 25.0)
PLACEBO_PRIOR_HL_SHIFT_ATR = (0.25, 1.00)
BOOTSTRAP_RESAMPLES = 10_000
BOOTSTRAP_UNIT = "hari perdagangan (day-cluster)"
CI_METHOD = "persentil 95%"

print("PREREGISTRASI PLACEBO Q6 (dikunci; dilarang tuning setelah nb04 dijalankan)")
for name, value in [
    ("Seed", PLACEBO_SEED),
    ("Timeframe", PLACEBO_TIMEFRAME),
    ("ATR window", TOUCH_ATR_WINDOW),
    ("Theta ATR", REACTION_THETA_ATR),
    ("Window k (bar M5)", REACTION_WINDOWS_BARS),
    ("Window primer", REACTION_WINDOW_PRIMARY),
    ("Offset placebo (pip)", PLACEBO_RANDOM_OFFSET_PIPS),
    ("Shift prior-HL (x ATR D1)", PLACEBO_PRIOR_HL_SHIFT_ATR),
    ("Bootstrap", f"{BOOTSTRAP_RESAMPLES:,} resample, {BOOTSTRAP_UNIT}"),
    ("CI", CI_METHOD),
]:
    print(f"  {name}: {value}")
assert PLACEBO_SEED == 20261006 and REACTION_WINDOW_PRIMARY in REACTION_WINDOWS_BARS

PREREGISTRASI PLACEBO Q6 (dikunci; dilarang tuning setelah nb04 dijalankan)
  Seed: 20261006
  Timeframe: M5
  ATR window: 14
  Theta ATR: 0.5
  Window k (bar M5): (6, 12, 24)
  Window primer: 12
  Offset placebo (pip): (5.0, 25.0)
  Shift prior-HL (x ATR D1): (0.25, 1.0)
  Bootstrap: 10,000 resample, hari perdagangan (day-cluster)
  CI: persentil 95%


In [11]:
checks = [
    (
        "Strata A/B/C = 1.136.401 / 185.248 / 26.198",
        (n_a, n_b, n_c) == (1_136_401, 185_248, 26_198),
    ),
    ("A+B+C = TOTAL 1.347.847", n_a + n_b + n_c == TOTAL == 1_347_847),
    (
        "Window C = 6.899 (5 hari) + 19.299 (14 hari)",
        [(bars, days) for _, _, bars, days in c_per_window] == [(6_899, 5), (19_299, 14)],
    ),
    ("H1-2023 = 130 hari berbar", days_b == 130),
    ("2025 = tahun referensi bersih (0 bar anomali)", bars_2025_anomaly == 0),
    ("tzdata == 2026.5", importlib.metadata.version("tzdata") == "2026.5"),
    ("D1 2024 = 260 bucket + nol bucket weekend", d1_ok),
    ("Semua TF: file lengkap, kolom wajib, 0 NaN", nan_ok),
    ("utc_series == parse(time) penuh M1 2025", time_eq_ok),
    (
        "4 zona DST-aware + 4 blok cross-tab terkunci",
        len(SESSION_SPECS) == 4 and len(CROSS_TAB_HOURS) == 4,
    ),
    ("Protokol placebo Q6 + seed terkunci", PLACEBO_SEED == 20261006),
]
for name, ok in checks:
    print(("PASS " if ok else "FAIL ") + name)
assert all(ok for _, ok in checks)
print()
print(
    f"RINGKASAN: {sum(ok for _, ok in checks)}/{len(checks)} audit PASS; strata & definisi terkunci."
)

PASS Strata A/B/C = 1.136.401 / 185.248 / 26.198
PASS A+B+C = TOTAL 1.347.847
PASS Window C = 6.899 (5 hari) + 19.299 (14 hari)
PASS H1-2023 = 130 hari berbar
PASS 2025 = tahun referensi bersih (0 bar anomali)
PASS tzdata == 2026.5
PASS D1 2024 = 260 bucket + nol bucket weekend
PASS Semua TF: file lengkap, kolom wajib, 0 NaN
PASS utc_series == parse(time) penuh M1 2025
PASS 4 zona DST-aware + 4 blok cross-tab terkunci
PASS Protokol placebo Q6 + seed terkunci

RINGKASAN: 11/11 audit PASS; strata & definisi terkunci.


## Penutup Notebook 00

**Terkunci**: strata A/B/C (+ angka & hari), definisi sesi primer/cross-tab + prioritas label,
schema & sumber kolom, definisi return/volatilitas/swing/level, aturan missing-data & anti look-ahead,
protokol placebo Q6 (parameter + seed) untuk notebook 04.

**Belum**: seluruh statistik EDA (return, volatilitas, spread, ACF, swing, level, regime) — itu notebook 01–05.
Kalender libur NYSE/TSE tetap di luar scope (missing-event dicatat di laporan).
Offset 2 window anomali tetap BELUM TERVERIFIKASI (strata C deskriptif saja).

**STOP** — notebook 01 menunggu review user atas notebook ini.